# AI Smart Mirror — Image Embedding Backfill (Colab GPU)

Upgrades the inventory catalog from **text-only** Marqo embeddings (name + color + category) to real **image** embeddings computed from actual product photos. Image embeddings capture color nuance, pattern, and silhouette that text alone can't — this is what makes Stage A retrieval (and the Stage B compatibility ranker) actually good.

**Runtime:** Runtime menu → Change runtime type → **T4 GPU** (free tier). ~10-15 minutes for the full ~37K-item catalog, vs ~2 hours on CPU.

**Before you run:** paste your Supabase credentials into Cell 3. **Delete that cell's contents (or the whole notebook) before sharing it with anyone** — the service_role key has full write access to your database.

In [ ]:
# Cell 1 — dependencies
!pip install open_clip_torch supabase datasets pillow -q

In [ ]:
# Cell 2 — confirm GPU is attached
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
# Cell 3 — CREDENTIALS -- paste from backend/.env, then DELETE this cell's contents before sharing the notebook
SUPABASE_URL = "https://YOUR-PROJECT.supabase.co"
SUPABASE_KEY = "YOUR-SERVICE-ROLE-KEY"

In [ ]:
# Cell 4 — load Marqo-FashionCLIP on GPU
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("hf-hub:Marqo/marqo-fashionCLIP")
model = model.to(device).eval()
print(f"Model ready on {device}")

In [ ]:
# Cell 5 — connect to Supabase and find rows still needing image embeddings
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

todo_by_name = {}
offset = 0
while True:
    r = sb.table("inventory").select("id,name").eq("embedding_source", "text").range(offset, offset + 999).execute()
    if not r.data:
        break
    for row in r.data:
        if row.get("name"):
            todo_by_name[row["name"]] = row["id"]
    if len(r.data) < 1000:
        break
    offset += 1000

print(f"{len(todo_by_name)} rows need image embeddings")

In [ ]:
# Cell 6 — load the source dataset (same one used to seed the catalog)
from datasets import load_dataset

ds = load_dataset("ashraq/fashion-product-images-small", split="train")
print(f"{len(ds)} rows in source dataset")

In [ ]:
# Cell 7 — the backfill loop: batch-embed images on GPU, bulk-write to Supabase
import time

BATCH_SIZE = 64  # T4 handles this comfortably; raise to 128 on an A100

def embed_batch(images):
    tensors = torch.stack([preprocess(img.convert("RGB")) for img in images]).to(device)
    with torch.no_grad():
        features = model.encode_image(tensors)
        features = features / features.norm(dim=-1, keepdim=True)
    return [f.tolist() for f in features.cpu()]

def retry(fn, tries=4, delay=3):
    last = None
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            print(f"  retry {i+1}/{tries}: {type(e).__name__}: {str(e)[:100]}")
            time.sleep(delay * (i + 1))
    raise last

batch_ids, batch_images = [], []
updated = 0
t0 = time.time()

def flush():
    global updated
    if not batch_ids:
        return
    embeddings = embed_batch(batch_images)
    retry(lambda: sb.rpc("bulk_update_embeddings", {"ids": batch_ids, "embeddings": embeddings}).execute())
    updated += len(batch_ids)
    elapsed = time.time() - t0
    rate = updated / elapsed if elapsed > 0 else 0
    eta_min = ((len(todo_by_name) - updated) / rate / 60) if rate > 0 else float("inf")
    print(f"  {updated}/{len(todo_by_name)}  ({elapsed:.0f}s, {rate:.1f} items/s, ETA {eta_min:.1f} min)")
    batch_ids.clear()
    batch_images.clear()

for row in ds:
    name = (row.get("productDisplayName") or "").strip()
    if name not in todo_by_name:
        continue
    img = row.get("image")
    if img is None:
        continue
    batch_ids.append(todo_by_name[name])
    batch_images.append(img)
    if len(batch_ids) >= BATCH_SIZE:
        flush()

flush()
print(f"\nDone: {updated} rows upgraded to real image embeddings in {time.time()-t0:.0f}s")

In [ ]:
# Cell 8 — sanity check: confirm the split
total = sb.table("inventory").select("id", count="exact").limit(1).execute().count
image_count = sb.table("inventory").select("id", count="exact").eq("embedding_source", "image").limit(1).execute().count
text_count = sb.table("inventory").select("id", count="exact").eq("embedding_source", "text").limit(1).execute().count
print(f"total: {total}")
print(f"image embeddings: {image_count}")
print(f"still text-only: {text_count}")

## Done

Your catalog now has real image embeddings. **Before closing this notebook:** clear Cell 3's credentials (Edit > Clear all outputs, then blank out the SUPABASE_URL/SUPABASE_KEY strings) if you plan to save or share this notebook.

No code changes needed on the backend — `inventory_search.py` reads whatever's in the `embedding` column, so retrieval quality improves immediately on your next query.